# Bayesian Analysis of Stochastic Volatility Models for Stock Returns

This notebook runs the full estimation and evaluation pipeline:
1. Load BRK-B stock data and compute log returns
2. Fit GARCH(1,1) baseline via maximum likelihood
3. Estimate Stochastic Volatility model via MCMC (Gibbs + Griddy-Gibbs)
4. Compare inferred volatilities
5. VaR backtesting: Kupiec and Christoffersen coverage tests

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import src
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['font.family'] = 'serif'
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
from src.data_loader import load_returns
from src.config import DataConfig

df, returns = load_returns()
print(f"Loaded {len(returns)} observations")
print(f"Date range: {df['Date'].iloc[0].date()} to {df['Date'].iloc[-1].date()}")
print(f"Mean return: {returns.mean():.4f}, Std: {returns.std():.4f}")

In [ ]:
from src.plotting import plot_returns, plot_acf

fig_dir = project_root / 'figures'

plot_returns(returns, ticker='BRK-B', save_path=fig_dir / 'log_returns.png')
plt.show()

plot_acf(returns, save_path=fig_dir / 'sample_acf.png')
plt.show()

## 2. GARCH(1,1) Baseline

In [ ]:
from src.garch_model import fit_garch, garch_residual_diagnostics

garch_result, garch_vol, garch_var = fit_garch(returns)
print(garch_result.summary())

diag = garch_residual_diagnostics(returns, garch_vol)
print(f"\nLjung-Box test on squared residuals: stat={diag['lb_stat']:.4f}, p={diag['lb_pvalue']:.4f}")

In [ ]:
from src.plotting import plot_inferred_volatility

plot_inferred_volatility(returns, garch_vol, model_name='GARCH(1,1)',
                         save_path=fig_dir / 'volatility_garch.png')
plt.show()

## 3. Stochastic Volatility Model (MCMC)

The SV model:
$$r_t = \sqrt{h_t} \, \varepsilon_t, \quad \varepsilon_t \sim N(0,1)$$
$$\ln h_t = \alpha_0 + \alpha_1 \ln h_{t-1} + v_t, \quad v_t \sim N(0, \sigma_v^2)$$

Estimation uses Gibbs sampling with Griddy-Gibbs for the latent log-volatilities.

In [ ]:
from src.sv_model import estimate_sv
from src.config import SVConfig

sv_config = SVConfig(n_gibbs=5000, seed=123)
print(f"Running MCMC with {sv_config.n_gibbs} iterations...")
sv_result = estimate_sv(returns, config=sv_config)

In [ ]:
# Posterior parameter summary
summary = sv_result.parameter_summary()
print("Posterior parameter estimates (after burn-in):")
for param, stats in summary.items():
    print(f"  {param}: mean = {stats['mean']:.4f}, std = {stats['std']:.4f}")

In [ ]:
from src.plotting import plot_traceplots

plot_traceplots(sv_result, save_path=fig_dir / 'traceplots.png')
plt.show()

In [ ]:
sv_vol = sv_result.posterior_mean_vol()

plot_inferred_volatility(returns, sv_vol, model_name='SV',
                         save_path=fig_dir / 'volatility_sv.png')
plt.show()

## 4. Volatility Comparison: SV vs GARCH

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(returns, linewidth=0.4, alpha=0.5, label='Log returns', color='gray')
ax.plot(garch_vol, linewidth=0.9, label='GARCH(1,1)', color='tab:blue')
ax.plot(sv_vol, linewidth=0.9, label='SV model', color='tab:red')
ax.set_xlabel('Trading days')
ax.set_ylabel(r'$r_t$, $\sqrt{h_t}$')
ax.set_title('Volatility Comparison: SV vs GARCH(1,1)')
ax.legend(loc='upper left')
ax.set_xlim(0, len(returns))
fig.tight_layout()
fig.savefig(fig_dir / 'volatility_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Value-at-Risk Backtesting

We compute 5% VaR estimates from both models and evaluate coverage using:
- **Kupiec (1995)** unconditional coverage test
- **Christoffersen (1998)** independence test
- **Conditional coverage** (combined) test

In [ ]:
from src.var_backtest import backtest_var, print_backtest_summary
from src.plotting import plot_var_backtest

# GARCH VaR backtest
garch_var_result = backtest_var(returns, garch_vol, alpha=0.05, model_name='GARCH(1,1)')
print_backtest_summary(garch_var_result)

plot_var_backtest(returns, garch_var_result, garch_vol,
                  save_path=fig_dir / 'var_backtest_garch.png')
plt.show()

In [ ]:
# SV model VaR backtest
sv_var_result = backtest_var(returns, sv_vol, alpha=0.05, model_name='SV')
print_backtest_summary(sv_var_result)

plot_var_backtest(returns, sv_var_result, sv_vol,
                  save_path=fig_dir / 'var_backtest_sv.png')
plt.show()

## 6. Summary Table

In [ ]:
import pandas as pd

rows = []
for res in [garch_var_result, sv_var_result]:
    rows.append({
        'Model': res.model_name,
        'Violations': res.kupiec['n_violations'],
        'Violation Rate': f"{res.kupiec['violation_rate']:.4f}",
        'Kupiec LR': f"{res.kupiec['lr_statistic']:.4f}",
        'Kupiec p': f"{res.kupiec['p_value']:.4f}",
        'Christoff. LR': f"{res.christoffersen['lr_statistic']:.4f}",
        'Christoff. p': f"{res.christoffersen['p_value']:.4f}",
        'CC LR': f"{res.cc_test['lr_statistic']:.4f}",
        'CC p': f"{res.cc_test['p_value']:.4f}",
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))